# 02 — Dense baseline analysis

Every compressed result is measured against its own model's dense FP32 baseline, so the
baselines have to be right before anything else is believed.

Checks:

1. Does each model's measured parameter count match the registry?
2. Does perplexity fall monotonically with model size, as the Pythia suite should?
3. Does CPU latency scale roughly linearly with parameter count?
4. Do all baseline benchmarks share one CPU and one thread count?

A failure in (2) usually means the evaluation windows differ between models — check the
`dataset_fingerprint` on each record.

In [ ]:
from scale_aware_compression.models.registry import registry_table, scale_sweep_models
from scale_aware_compression.visualisation.tables import rows_to_markdown

print(rows_to_markdown(registry_table()))
print()
print("sweep order:", scale_sweep_models(include_optional=True))

In [ ]:
from scale_aware_compression.experiments.runner import ExperimentTracker

tracker = ExperimentTracker("../outputs/metrics")
records = tracker.load_all()
dense = [record for record in records if record.get("compression_method") == "dense"]
print(f"{len(records)} record(s) total, {len(dense)} dense baseline(s)")

In [ ]:
# Comparability check. More than one CPU model or thread count means these rows must not share
# a figure.
machines = {record.get("hardware", {}).get("cpu_model") for record in dense}
threads = {record.get("deployment", {}).get("num_threads") for record in dense}
fingerprints = {
    record.get("quality", {}).get("perplexity", {}).get("dataset_fingerprint")
    for record in dense
}
print("CPU models:", machines)
print("thread counts:", threads)
print("evaluation fingerprints:", fingerprints)
assert len(machines) <= 1, "baselines span multiple machines; latencies are not comparable"
assert len(threads) <= 1, "baselines span multiple thread counts"

In [ ]:
rows = [
    {
        "model": record["model_name"],
        "params": record.get("parameter_count"),
        "perplexity": record.get("quality", {}).get("perplexity", {}).get("perplexity"),
        "median_ms": record.get("deployment", {}).get("latency_median_ms"),
        "p95_ms": record.get("deployment", {}).get("latency_p95_ms"),
        "tok_per_s": record.get("deployment", {}).get("throughput_tokens_per_s"),
        "peak_mb": record.get("deployment", {}).get("peak_memory_mb"),
    }
    for record in sorted(dense, key=lambda item: item.get("parameter_count") or 0)
]
print(rows_to_markdown(rows) if rows else "No dense baselines recorded yet.")

## Still to do

- plot perplexity against parameter count on log axes and check the trend is monotone
- plot median latency against parameter count and fit the slope, as the cost reference for the
  compressed arms
- inspect the per-run latency distributions for bimodality, which indicates thermal throttling
  part-way through a measurement